# Semaine 7 — Jour 3 : Événements, contrats et idempotence

Objectif : Distinguer événement, commande et réponse HTTP.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J3_evenements_contrats.ipynb` et `notebooks/week07/teacher/S7_J3_evenements_contrats_teacher.ipynb`.

Critère de réussite : Expliquer les limites d’une livraison au moins une fois.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Distinguer événement, commande et réponse HTTP.
- Définir un contrat versionné et détecter la réutilisation abusive d’un identifiant.
- Expliquer les limites d’une livraison au moins une fois.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 3 : Événements, contrats et idempotence


## 1. Un événement représente un fait
`support.requested` indique qu’une demande a été reçue par le système amont. Le fait garde le même identifiant lors d’une nouvelle livraison. Une commande comme « préparer un brouillon » exprime une intention et peut être refusée. Une réponse HTTP indique le résultat d’un échange technique, sans devenir automatiquement un événement métier.

Notre enveloppe minimale possède `event_id`, `schema_version`, `type`, `ticket_id` et `text`. Elle n’implémente pas CloudEvents; ce standard serait une option d’interopérabilité. `ticket_id` corrèle plusieurs faits pour une même demande, tandis que `event_id` distingue les faits. Ajouter une correction à un ticket exige un nouvel événement, pas une nouvelle livraison du précédent.

## 2. Le contrat est une frontière
Le validateur exige des champs connus, une version entière égale à 1 et un type reconnu. Les valeurs textuelles sont non vides et bornées. Même une valeur JSON `true` est refusée pour la version : Python considère normalement bool comme une sous-classe de int, ce qui peut rendre un contrôle naïf trop permissif.

Un contrat strict révèle tôt une incompatibilité. Il exige aussi une politique de migration. Si un producteur ajoute un champ, ce consommateur version 1 le refusera. L’exercice vise à rendre le comportement explicite, pas à prescrire le rejet de tous les champs supplémentaires dans toutes les architectures. Une évolution compatible peut choisir d’ignorer certains champs, à condition de le documenter et de le tester.

## 3. Au moins une fois signifie doublons possibles
Le producteur peut perdre la réponse après que le backend a commité. Il ne sait pas si son événement a été accepté et le renvoie. Le réseau ne fournit pas une garantie métier « exactement une fois ». Le backend compare l’identifiant et le contenu canonique, puis retourne le job existant pour un doublon identique.

La clé primaire `event_id` protège le stockage. Le contenu sérialisé avec des clés triées neutralise l’ordre des clés JSON. Si le même identifiant est envoyé avec un autre texte, le service répond 409. Supprimer simplement le doublon sans comparer son contenu masquerait un bug du producteur ou une collision d’identifiants.

## 4. Déduplication et idempotence ne se confondent pas
La déduplication reconnaît une livraison déjà reçue. L’idempotence garantit qu’une répétition admissible n’ajoute pas un nouvel effet. La publication locale utilise une clé primaire sur event_id et une transition transactionnelle. Deux demandes distinctes avec deux identifiants peuvent toujours publier deux réponses pour le même ticket : c’est un problème métier différent. Une politique par ticket doit être définie si ce comportement est indésirable.

Ne promettez pas un « exactly once » universel à partir de cette table. Un fournisseur de messagerie externe peut accepter l’envoi puis perdre la connexion avant l’accusé de réception. Il faut un identifiant d’effet stable, une idempotency key supportée par le fournisseur, ou une réconciliation. Sans ce support, une livraison répétée peut produire un doublon externe.

## 5. Accusé de réception et durée de rétention
On répond après la persistance du fait. Dans le lab, les jobs et les identifiants restent conservés; il n’y a pas de purge des jobs métier. Une installation réelle doit définir combien de temps un identifiant reste dédupliquable. Purger trop tôt permet à une livraison tardive de redevenir un événement neuf.

## À retenir
Un event_id stable rend une nouvelle livraison reconnaissable. Il ne résout ni l’ordre des événements d’un ticket, ni l’atomicité avec un système externe, ni toutes les règles métier de duplication.


## Diagramme du jour

```mermaid
flowchart TD
  A["Événement reçu"] --> B{"Identifiant connu ?"}
  B -->|non| C["Commit puis 202"]
  B -->|oui| D{"Contenu identique ?"}
  D -->|oui| E["Job existant et 200"]
  D -->|non| F["Conflit 409"]
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
import json
left = {"event_id": "e1", "text": "Bonjour"}
right = {"text": "Bonjour", "event_id": "e1"}
def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False)
assert canonical(left) == canonical(right)
assert canonical(left) != canonical({**left, "text": "Correction"})
print("Ordre des clés neutralisé, collision de contenu détectable")


# Lab J3 — Événements, contrats et idempotence

Depuis la racine du dépôt :

```bash
python3 book/week07/day03/labs/run.py
```

Résultat attendu : **duplicate_created=false et un seul job received**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Dessiner une enveloppe

Fournir un exemple valide puis une version incompatible.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Simuler une livraison répétée

Envoyer la même donnée deux fois, puis modifier text en conservant event_id. Quels résultats attendre ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Analyser deux identifiants

Deux event_id différents pour le même ticket doivent-ils être fusionnés ? Justifier.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Que garantit une clé primaire de déduplication ?

## Question 2

Quelle différence entre corrélation et identité ?


# Challenge

Ajouter dans une branche d’exercice un event_id avec un autre ticket_id et vérifier que la collision est refusée. Puis envoyer un nouvel event_id pour le même ticket et expliquer l’acceptation.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J3

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.


# Corrigé des exercices

## Exercice 1 — Dessiner une enveloppe

L’exemple assets/event.json est valide. schema_version:2 est refusé. Un schéma version 2 doit avoir un consommateur capable de le comprendre et une migration explicite.

## Exercice 2 — Simuler une livraison répétée

Premier POST:202; répétition identique:200; contenu différent:409. Le premier job reste inchangé.

## Exercice 3 — Analyser deux identifiants

Pas automatiquement. Il peut s’agir de deux faits distincts. Une règle métier de supersession ou de regroupement par ticket est nécessaire avant de fusionner.


# Réponses d’entretien

## Question 1

Elle empêche plusieurs lignes pour la même clé dans ce stockage. Elle ne garantit pas une écriture externe unique. En concurrence, la gestion des conflits de transaction doit aussi être définie.

## Question 2

La corrélation relie plusieurs faits d’un processus; l’identité distingue un fait. Ici ticket_id corrèle et event_id identifie.


# Solution du challenge

Le payload canonique contient ticket_id : modifier ce champ avec le même event_id entraîne Conflict. Avec event_id neuf, le job est accepté. Cette différence montre que déduplication technique et politique métier par ticket sont distinctes.


# Revue pédagogique

## Acquis à vérifier

- Distinguer événement, commande et réponse HTTP.
- Définir un contrat versionné et détecter la réutilisation abusive d’un identifiant.
- Expliquer les limites d’une livraison au moins une fois.

## Erreurs fréquentes

Confondre réception et achèvement; assimiler un exemple simulé à une évaluation de modèle; déduire une garantie distribuée d’une transaction locale. Pour ce jour, vérifier en particulier : Pas automatiquement. Il peut s’agir de deux faits distincts. Une règle métier de supersession ou de regroupement par ticket est nécessaire avant de fusionner.

## Barème indicatif sur 10

- 3 points : explication des concepts avec le scénario du support.
- 3 points : reproduction du lab et analyse de sa sortie.
- 2 points : réponses aux exercices, sans confusion sur les garanties.
- 2 points : challenge argumenté et limite opérationnelle identifiée.

Le parcours essentiel peut être validé sans le challenge. Toute publication sans validation bloque la validation du projet intégré, quel que soit le score.


# Correction exécutable du lab

Dans Jupyter, démarrer le kernel depuis la racine du dépôt ou depuis son arbre. Ce fragment recherche la racine contenant le module S7. Il utilise le moteur complet, distinct du petit exemple autonome.



In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/week07/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis le dépôt décompressé")
sys.path.insert(0, str(root))
from examples.week07.demo import scenario
result = scenario(3)
assert result == {'duplicate_created': False, 'jobs': {'received': 1}}
print(result)



Les assertions constituent la preuve minimale de ce scénario; les tests de `tests/week07` couvrent les branches complémentaires. Le payload canonique contient ticket_id : modifier ce champ avec le même event_id entraîne Conflict. Avec event_id neuf, le job est accepté. Cette différence montre que déduplication technique et politique métier par ticket sont distinctes.
